# Credit Card Fraud Detection: Professional Walkthrough

This notebook uses the required neural network to detect fraudulent (`Class=1`) transactions in Kaggle's highly imbalanced credit-card dataset. It adds a chronological evaluation design and two focused baselines so the results are interpreted against useful context rather than accuracy alone.

## Course rubric map

| Rubric block | Points | Notebook sections |
|---|---:|---|
| Problem and dataset | 2 | 1–2: objective, provenance, and schema |
| Exploratory analysis | 3 | 4: training-only class balance and plots |
| Preprocessing and neural network | 4 | 5–8: chronological split, scaling, baselines, MLP, and training |
| Threshold and evaluation | 2 | 9–10: validation thresholds and frozen test report |
| Interpretation and limitations | 3 | 11: evidence-led discussion |
| Reproducibility and delivery | 1 | Run instructions and delivery checklist in `professional/README.md` |

Use the assignment rubric's exact wording in your presentation. This notebook starts with empty outputs: do not claim results until you run it and can point to the observed tables and plots.

## 1. Problem, data source, and evaluation plan

The task is binary classification: `Class=0` is a legitimate transaction and `Class=1` is fraud. The public dataset is `mlg-ulb/creditcardfraud`, file `creditcard.csv`, with `Time`, `V1`–`V28`, and `Amount` as numeric inputs. Only about 0.172% of the commonly reported full dataset is fraud, so an always-legitimate classifier can have high accuracy while detecting no fraud. The notebook calculates all observed counts from the downloaded file rather than assuming fixed row totals.

The primary split is chronological: approximately 65% earliest rows for training, the next 15% for validation, and the latest 20% for final testing. This better represents scoring later transactions than random stratification. Equal timestamps stay together when choosing nearby boundaries; the actual proportions are reported. Since the dataset covers only about two days in 2013, this is a limited historical time holdout, not evidence of current or long-term performance.

The only model families are a majority-class dummy, a balanced logistic-regression benchmark, and the assignment's Keras MLP. The two baselines contextualize performance and the accuracy trap; they do not replace the neural network required by the assignment.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_detection.data import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    download_credit_card_csv,
    load_credit_card_csv,
)
from fraud_detection.evaluation import (
    evaluate_binary_classifier,
    positive_class_probabilities,
    select_f1_threshold,
)
from fraud_detection.models import build_logistic_regression, build_majority_baseline
from fraud_detection.models.neural_network import (
    build_mlp,
    compute_training_class_weights,
    make_early_stopping,
)
from fraud_detection.preprocessing import scale_splits
from fraud_detection.splitting import chronological_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter with professional/ as the working directory.')

DATA_PATH = PROJECT_ROOT / 'data' / 'creditcard.csv'
print(f'Project directory: {PROJECT_ROOT}')
print(f'Dataset path: {DATA_PATH}')

## 2. Download and validate the dataset

If the local CSV is absent, KaggleHub requests exactly `dataset_download('mlg-ulb/creditcardfraud', path='creditcard.csv')` and stages the returned file under `professional/data/creditcard.csv`. Internet access is required. Kaggle may require user consent for this public resource; configure any necessary authentication through KaggleHub outside the notebook. Never put tokens or credentials in project files. The loader checks the required numeric columns, finite values, and binary target before analysis.

In [ ]:
if not DATA_PATH.is_file():
    download_credit_card_csv(DATA_PATH)

data = load_credit_card_csv(DATA_PATH)
print(f'Observed shape: {data.shape[0]:,} rows by {data.shape[1]} columns')
print(f'Input feature count: {len(FEATURE_COLUMNS)}')
display(data.loc[:, list(FEATURE_COLUMNS)].head())

In [ ]:
feature_data = data.loc[:, list(FEATURE_COLUMNS)]
data_quality = pd.DataFrame(
    {
        'dtype': feature_data.dtypes.astype(str),
        'missing_values': feature_data.isna().sum(),
        'unique_values': feature_data.nunique(dropna=False),
    }
)
display(data_quality)
print(f'Rows with any missing input feature: {int(feature_data.isna().any(axis=1).sum()):,}')
print(f'Rows duplicated across input features: {int(feature_data.duplicated().sum()):,}')

## 3. Chronological train / validation / test split

Boundary selection uses only `Time` and row counts, never `Class`. The helper sorts stably and selects the nearest feasible timestamp-group boundaries to the requested proportions, so identical timestamps are not divided across partitions. Validation labels are reserved for early stopping and threshold/model comparison; test labels are not inspected until the final evaluation. The reported realized row shares may differ slightly from 65/15/20 because tied timestamps are kept intact.

In [ ]:
X = data.loc[:, list(FEATURE_COLUMNS)].copy()
y = data[TARGET_COLUMN].astype('int32').copy()

split = chronological_split(
    X,
    y,
    time_column='Time',
    train_fraction=0.65,
    validation_fraction=0.15,
)
X_train, y_train = split.X_train, split.y_train
X_validation, y_validation = split.X_validation, split.y_validation
X_test, y_test = split.X_test, split.y_test

split_summary = pd.DataFrame(
    [
        {
            'partition': name,
            'rows': len(features),
            'row_proportion': len(features) / len(X),
            'earliest_Time': features['Time'].min(),
            'latest_Time': features['Time'].max(),
        }
        for name, features in (
            ('Train', X_train),
            ('Validation', X_validation),
            ('Test', X_test),
        )
    ]
)
display(split_summary.set_index('partition'))

## 4. Exploratory data analysis: training subset only

All label-based counts, class-conditioned transaction plots, and feature-to-target correlations below use `X_train` and `y_train` only. This avoids using validation/test labels for exploratory analysis. The rare class is expected to be a tiny share of the data, but report the prevalence observed in this training partition instead of hardcoding a count. `Time` is seconds from the first recorded transaction, not a calendar timestamp. These historical training plots do not establish future performance.

In [ ]:
training_class_counts = y_train.value_counts().reindex([0, 1], fill_value=0)
training_class_summary = pd.DataFrame(
    {
        'transactions': training_class_counts,
        'percentage': 100 * training_class_counts / len(y_train),
    }
)
display(training_class_summary.rename(index={0: 'Legitimate (0)', 1: 'Fraud (1)'}))

train_eda = X_train.copy()
train_eda['Class'] = y_train
train_eda['Amount_log1p'] = np.log1p(train_eda['Amount'])
train_eda['Time_hours'] = train_eda['Time'] / 3600
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.barplot(
    x=['Legitimate (0)', 'Fraud (1)'],
    y=training_class_counts.to_numpy(),
    color='#4472C4',
    ax=axes[0],
)
axes[0].set(title='Training class balance', xlabel='Class', ylabel='Transactions')
sns.histplot(
    data=train_eda, x='Amount_log1p', hue='Class', stat='density',
    common_norm=False, element='step', fill=False, bins=50, ax=axes[1]
)
axes[1].set(title='Amount by class (training only)', xlabel='log(1 + Amount)', ylabel='Class-conditional density')
sns.histplot(
    data=train_eda, x='Time_hours', hue='Class', stat='density',
    common_norm=False, element='step', fill=False, bins=50, ax=axes[2]
)
axes[2].set(title='Transaction time by class (training only)', xlabel='Hours from dataset start', ylabel='Class-conditional density')
plt.tight_layout()
plt.show()

In [ ]:
training_correlations = (
    train_eda.loc[:, list(FEATURE_COLUMNS) + ['Class']]
    .corr(numeric_only=True)['Class']
    .drop('Class')
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)
display(training_correlations.rename('absolute_correlation_with_Class').to_frame())
ax = training_correlations.plot(kind='barh', figsize=(8, 5), color='#70AD47')
ax.set(
    title='Largest absolute feature correlations with Class (training only)',
    xlabel='Absolute Pearson correlation',
    ylabel='Feature',
)
plt.tight_layout()
plt.show()

## 5. Scale features without leakage

The 30 inputs have different scales: `Time` and `Amount` are not on the same scale as the anonymized PCA-derived `V1`–`V28`. A dense network benefits from standardized inputs. The scaler below is fitted on `X_train` only; validation and test features are transformed with that fitted scaler and never affect its means or standard deviations. The logistic baseline has its own `StandardScaler` inside a scikit-learn `Pipeline`, which is fitted only on training rows. No validation or test rows are resampled.

In [ ]:
scaled = scale_splits(X_train, X_validation, X_test)
print(f'Scaled train shape: {scaled.X_train.shape}')
print(f'Scaled validation shape: {scaled.X_validation.shape}')
print(f'Scaled test shape: {scaled.X_test.shape}')
print('Scaler fit rows:', int(scaled.scaler.n_samples_seen_))

## 6. Focused baselines

The `DummyClassifier(strategy='most_frequent')` exposes the accuracy trap: if fraud is rare, predicting only the majority class can look accurate and still have zero fraud recall. The balanced logistic-regression pipeline is a conventional, train-scaled benchmark. These models provide context for the neural network required by the assignment; neither is a substitute for it. Both are fitted on training data only.

In [ ]:
majority_model = build_majority_baseline()
logistic_model = build_logistic_regression(random_state=SEED)

majority_model.fit(X_train, y_train)
logistic_model.fit(X_train, y_train)

majority_validation_probabilities = positive_class_probabilities(
    majority_model, X_validation
)
logistic_validation_probabilities = positive_class_probabilities(
    logistic_model, X_validation
)
print('Both baselines were fitted on training rows only.')

## 7. Build and train the required neural network

The MLP has 64 ReLU units, Dropout 0.2, 32 ReLU units, another Dropout 0.2 layer, and a one-unit sigmoid output for fraud probability. ReLU provides nonlinear hidden representations; dropout randomly omits hidden activations during training as a regularizer. Binary cross-entropy is the binary classification loss, and Adam is the optimizer. The notebook caps training at 50 epochs with batch size 2,048, while EarlyStopping monitors validation PR-AUC with patience 5 and restores the best weights.

Class weights are computed from `y_train` only, making mistakes on the rare fraud class matter more to the training loss. Weighting does not create new information or guarantee calibrated probabilities. Validation labels drive early stopping; test labels do not. TensorFlow is imported lazily inside the model helper, so the offline utility tests do not need to import TensorFlow.

In [ ]:
class_weights = compute_training_class_weights(y_train)
print(f'Training-only class weights: {class_weights}')

mlp_model = build_mlp(input_features=scaled.X_train.shape[1], seed=SEED)
mlp_model.summary()

early_stopping = make_early_stopping(patience=5)
history = mlp_model.fit(
    scaled.X_train,
    y_train.to_numpy(),
    validation_data=(scaled.X_validation, y_validation.to_numpy()),
    class_weight=class_weights,
    epochs=50,
    batch_size=2048,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_frame['loss'], label='Training')
axes[0].plot(history_frame['val_loss'], label='Validation')
axes[0].set(title='Binary cross-entropy', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(history_frame['pr_auc'], label='Training Keras PR-AUC')
axes[1].plot(history_frame['val_pr_auc'], label='Validation Keras PR-AUC')
axes[1].set(title='Keras precision-recall AUC', xlabel='Epoch', ylabel='PR-AUC')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Epochs completed: {len(history_frame)}')

## 8. Compare models on validation Average Precision

Average Precision (AP) from scikit-learn is the primary validation comparison because it summarizes precision-recall ranking under rare positives. It is a step-wise summary, not the trapezoidal area under a plotted PR curve. The Keras PR-AUC above is an approximate threshold-based training metric used for early stopping; it is not substituted for scikit-learn AP. The dummy model's constant positive-class score is a useful ranking reference even though its operational prediction remains the majority class.

In [ ]:
mlp_validation_probabilities = mlp_model.predict(
    scaled.X_validation, verbose=0
).ravel()

validation_probabilities = {
    'Most-frequent dummy': majority_validation_probabilities,
    'Balanced logistic regression': logistic_validation_probabilities,
    'Keras MLP': mlp_validation_probabilities,
}
validation_average_precision = {
    name: evaluate_binary_classifier(
        y_validation, probabilities, threshold=0.5
    ).average_precision
    for name, probabilities in validation_probabilities.items()
}
validation_comparison = pd.DataFrame.from_dict(
    validation_average_precision, orient='index', columns=['Average Precision (sklearn)']
)
display(validation_comparison.sort_values('Average Precision (sklearn)', ascending=False))

## 9. Select demonstration thresholds using validation only

A probability needs a decision threshold before it becomes a class label. Choose the logistic and MLP thresholds by maximizing validation F1, then freeze them before test evaluation. F1 is only a teaching demonstration: it treats precision and recall symmetrically and does not encode transaction values, investigation capacity, or the cost of a false negative. The majority baseline is intentionally not threshold-tuned; it remains an always-most-frequent classifier. If validation happens to contain only one class, F1 threshold selection is not meaningful and the helper stops with a clear error instead of silently choosing a threshold.

In [ ]:
logistic_threshold = select_f1_threshold(
    y_validation, logistic_validation_probabilities
)
mlp_threshold = select_f1_threshold(y_validation, mlp_validation_probabilities)

validation_threshold_report = {
    'Balanced logistic regression': evaluate_binary_classifier(
        y_validation, logistic_validation_probabilities, threshold=logistic_threshold
    ),
    'Keras MLP': evaluate_binary_classifier(
        y_validation, mlp_validation_probabilities, threshold=mlp_threshold
    ),
}
display(
    pd.DataFrame(
        {
            name: {
                'frozen_threshold': metrics.threshold,
                'precision': metrics.precision,
                'recall': metrics.recall,
                'F1': metrics.f1,
            }
            for name, metrics in validation_threshold_report.items()
        }
    ).T
)

## 10. Final held-out test evaluation

No model, preprocessing step, or threshold is changed after seeing test labels. Apply the train-fitted scaler to test features, score each model, and report results once using the frozen validation thresholds. Average Precision is the primary ranking metric; ROC-AUC is secondary. Precision, recall, F1, and the confusion matrix describe the operating point. A test partition without both classes has undefined ROC-AUC; it is reported as undefined rather than fabricated. If there are no test fraud examples, Average Precision is also left undefined by the evaluation helper. The majority baseline's accuracy and fraud recall are reported together to make the accuracy trap concrete.

In [ ]:
majority_test_probabilities = positive_class_probabilities(majority_model, X_test)
logistic_test_probabilities = positive_class_probabilities(logistic_model, X_test)
mlp_test_probabilities = mlp_model.predict(scaled.X_test, verbose=0).ravel()

test_probabilities = {
    'Most-frequent dummy': majority_test_probabilities,
    'Balanced logistic regression': logistic_test_probabilities,
    'Keras MLP': mlp_test_probabilities,
}
frozen_thresholds = {
    'Most-frequent dummy': 0.5,
    'Balanced logistic regression': logistic_threshold,
    'Keras MLP': mlp_threshold,
}
test_metrics = {
    name: evaluate_binary_classifier(
        y_test, probabilities, threshold=frozen_thresholds[name]
    )
    for name, probabilities in test_probabilities.items()
}

test_summary = pd.DataFrame(
    {
        name: {
            'frozen_threshold': metrics.threshold,
            'accuracy': metrics.accuracy,
            'Average Precision (sklearn)': metrics.average_precision,
            'ROC-AUC (secondary)': metrics.roc_auc,
            'precision': metrics.precision,
            'recall': metrics.recall,
            'F1': metrics.f1,
        }
        for name, metrics in test_metrics.items()
    }
).T
display(test_summary)

majority_class = int(majority_model.classes_[np.argmax(majority_model.class_prior_)])
majority_result = test_metrics['Most-frequent dummy']
print(f'Majority class learned from training data: {majority_class}')
print(f'Majority baseline test accuracy: {majority_result.accuracy:.4f}')
print(f'Majority baseline test fraud recall: {majority_result.recall:.4f}')

for name, metrics in test_metrics.items():
    display(
        pd.DataFrame(
            metrics.confusion_matrix,
            index=['Actual legitimate (0)', 'Actual fraud (1)'],
            columns=['Predicted legitimate (0)', 'Predicted fraud (1)'],
        ).style.set_caption(f'{name}: test confusion matrix')
    )
    if np.isnan(metrics.roc_auc):
        print(f'{name}: ROC-AUC is undefined because the test partition does not contain both classes.')
    if np.isnan(metrics.average_precision):
        print(f'{name}: Average Precision is undefined because the test partition has no fraud examples.')

## 11. Interpretation, trade-offs, and limitations

After running the notebook, write a short conclusion from the observed outputs above; do not insert expected or invented scores. Use the test confusion matrices to explain how many legitimate transactions were flagged and how many frauds were missed. Compare the majority baseline's accuracy with its fraud recall. Explain what scikit-learn Average Precision and secondary ROC-AUC show beyond the threshold-specific precision, recall, and F1. Discuss whether the false-alarm versus missed-fraud trade-off would be acceptable for the stated use case.

Keep these limits explicit: the data represents only about two days in 2013, so a chronological holdout is more future-like than a random split but still too short to establish robust temporal generalization; `V1`–`V28` are anonymized PCA-derived features and do not support straightforward semantic explanations; class weighting cannot repair missing information; and validation-selected F1 does not encode real business or operational costs. Production use would need a cost-based policy, monitoring, and evaluation on later data.